In [0]:
from pyspark.sql import functions as F

base  = "/Volumes/capstone6_telecom/telecom/ext_raw_landing/adls/"
table = "capstone6_telecom.telecom.bronze_network_events"

In [0]:
df = (spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("header", "true")
      .option("cloudFiles.schemaLocation", f"{base}/_schemas/primary_events")
      .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
      .option("cloudFiles.inferColumnTypes", "true")
      .option("cloudFiles.schemaHints", "latency_ms DOUBLE, packet_loss_pct DOUBLE")
      .load(f"{base}/primary_events/")
      .withColumn("source_file", F.col("_metadata.file_path"))
      .withColumn("ingest_ts", F.current_timestamp()))

(df.writeStream
   .option("checkpointLocation", f"{base}/_checkpoints/primary_events")
   .option("mergeSchema", "true")
   .trigger(availableNow=True)
   .toTable(table)
   .awaitTermination())

In [0]:
%sql
SELECT COUNT(*) AS total_rows FROM capstone6_telecom.telecom.bronze_network_events

In [0]:
%sql
SELECT source_file, COUNT(*) AS rows
FROM capstone6_telecom.telecom.bronze_network_events
GROUP BY source_file
ORDER BY source_file